# 00 — Data Audit

**ML Perovskites project — raw-data audit only**

Purpose: inspect the two raw input datasets before any feature engineering or modeling.

**This notebook does not:**
- create ML features;
- impute missing values;
- remove correlated variables;
- train models;
- create train/test splits.

It only documents what is actually present in the source files and flags issues that must be handled later.

In [0]:
# ============================================================
# 1. CONFIGURATION
# ============================================================

DB_ML_TABLE = "ml_perovskites.db_ml"
ATOM_PROPS_TABLE = "ml_perovskites.atom_props"

OUTPUT_SCHEMA = "ml_perovskites"

print("db_ml:", DB_ML_TABLE)
print("atom_props:", ATOM_PROPS_TABLE)

In [0]:
# ============================================================
# 2. IMPORTS AND HELPERS
# ============================================================

import re
import json
import numpy as np
import pandas as pd

from pyspark.sql import functions as F


def clean_column_names(df):
    """
    Clean column names while preserving the original dataframe type.
    """
    return df.toDF(*[
        str(c).strip().replace("\ufeff", "")
        for c in df.columns
    ])


def find_col(df, candidates):
    """
    Find a column using case-insensitive matching.
    """
    lookup = {
        str(c).strip().lower(): c
        for c in df.columns
    }

    for candidate in candidates:
        if candidate.lower() in lookup:
            return lookup[candidate.lower()]

    return None


print("Imports and helper functions loaded.")

In [0]:
# ============================================================
# 3. LOAD RAW DATA FROM DATABRICKS TABLES
# ============================================================

DB_ML_TABLE = "ml_perovskites.db_ml"
ATOM_PROPS_TABLE = "ml_perovskites.atom_props"

# db_ml is small (~300 rows), so Pandas is fine for the audit.
db_ml = (
    spark.table(DB_ML_TABLE)
    .toPandas()
)

# atom_props is large.
# Keep it as a Spark DataFrame.
atom_props_spark = spark.table(ATOM_PROPS_TABLE)

print(f"db_ml shape:       {db_ml.shape}")
print(
    f"atom_props shape:  "
    f"({atom_props_spark.count():,}, "
    f"{len(atom_props_spark.columns)})"
)

## 4. Dataset-level overview

In [0]:
# ============================================================
# 4. DATASET-LEVEL OVERVIEW
# ============================================================

db_ml_rows = db_ml.shape[0]
db_ml_cols = db_ml.shape[1]

atom_props_rows = atom_props.count()
atom_props_cols = len(atom_props.columns)

print("DATASET OVERVIEW")
print("-" * 50)

print(f"db_ml:")
print(f"  Rows:    {db_ml_rows}")
print(f"  Columns: {db_ml_cols}")
print(f"  Exact duplicate rows: {db_ml.duplicated().sum()}")

print()

print(f"atom_props:")
print(f"  Rows:    {atom_props_rows}")
print(f"  Columns: {atom_props_cols}")
print(f"  Exact duplicate rows: not calculated at this stage")

In [0]:
# ============================================================
# 5. COLUMN INVENTORY — db_ml
# ============================================================

db_inventory = pd.DataFrame({
    "column": db_ml.columns,
    "dtype": [str(db_ml[c].dtype) for c in db_ml.columns],
    "non_null": [db_ml[c].notna().sum() for c in db_ml.columns],
    "missing": [db_ml[c].isna().sum() for c in db_ml.columns],
    "missing_pct": [
        100 * db_ml[c].isna().mean()
        for c in db_ml.columns
    ],
    "n_unique": [
        db_ml[c].nunique(dropna=True)
        for c in db_ml.columns
    ]
})

display(db_inventory)

In [0]:
# ============================================================
# 5B. COLUMN INVENTORY — atom_props
# ============================================================

atom_props_spark = spark.table(ATOM_PROPS_TABLE)

atom_props_rows = atom_props_spark.count()

atom_inventory_rows = []

for field in atom_props_spark.schema.fields:

    c = field.name

    stats = (
        atom_props_spark
        .select(
            F.count(F.col(c)).alias("non_null"),
            F.countDistinct(F.col(c)).alias("n_unique")
        )
        .first()
    )

    non_null = stats["non_null"]

    atom_inventory_rows.append({
        "column": c,
        "dtype": str(field.dataType),
        "non_null": non_null,
        "missing": atom_props_rows - non_null,
        "missing_pct": (
            100 * (atom_props_rows - non_null)
            / atom_props_rows
        ),
        "n_unique": stats["n_unique"]
    })

atom_inventory = pd.DataFrame(atom_inventory_rows)

display(atom_inventory)

## 5. Missingness audit

In [0]:
# ============================================================
# 6. MISSINGNESS — db_ml
# ============================================================

db_missingness = (
    db_inventory
    .sort_values("missing_pct", ascending=False)
    .reset_index(drop=True)
)

display(db_missingness)

In [0]:
# ============================================================
# 6B. MISSINGNESS — atom_props
# ============================================================

atom_missingness = (
    atom_inventory
    .sort_values("missing_pct", ascending=False)
    .reset_index(drop=True)
)

display(atom_missingness)

## 6. Duplicate and repeated-experiment audit

In [0]:
# ============================================================
# 7. REPEATED EXPERIMENT AUDIT
# ============================================================

def find_col(df, candidates):

    lookup = {
        str(c).strip().lower(): c
        for c in df.columns
    }

    for candidate in candidates:
        if candidate.lower() in lookup:
            return lookup[candidate.lower()]

    return None


compound_col = find_col(
    db_ml,
    ["Compound", "compound"]
)

doi_col = find_col(
    db_ml,
    ["doi_reference", "doi"]
)

print("Compound column:", compound_col)
print("DOI column:", doi_col)


if compound_col is not None:

    compound_counts = (
        db_ml
        .groupby(compound_col, dropna=False)
        .size()
        .reset_index(name="experiment_count")
        .sort_values("experiment_count", ascending=False)
    )

    print(
        "Unique compounds:",
        db_ml[compound_col].nunique(dropna=True)
    )

    display(
        compound_counts[
            compound_counts["experiment_count"] > 1
        ].head(50)
    )

In [0]:
# ============================================================
# 7B. DUPLICATE COMPOUND / DOI GROUPS
# ============================================================

duplicate_key_cols = [
    c for c in [compound_col, doi_col]
    if c is not None
]

if duplicate_key_cols:

    duplicate_groups = (
        db_ml
        .groupby(
            duplicate_key_cols,
            dropna=False
        )
        .size()
        .reset_index(name="row_count")
        .query("row_count > 1")
        .sort_values("row_count", ascending=False)
    )

    display(
        duplicate_groups.head(50)
    )

## 7. Target audit

In [0]:
# ============================================================
# 8. TARGET AUDIT
# ============================================================

bandgap_col = find_col(
    db_ml,
    ["bandgap_energy"]
)

h2_col = find_col(
    db_ml,
    ["hydrogen_production_rate"]
)

print("Bandgap target:", bandgap_col)
print("Hydrogen target:", h2_col)

target_rows = []

for target_name, col in [
    ("bandgap_energy", bandgap_col),
    ("hydrogen_production_rate", h2_col)
]:

    if col is None:
        continue

    x = pd.to_numeric(
        db_ml[col],
        errors="coerce"
    )

    target_rows.append({
        "target": target_name,
        "column": col,
        "n_total": len(x),
        "n_valid": int(x.notna().sum()),
        "n_missing": int(x.isna().sum()),
        "n_zero": int((x == 0).sum()),
        "n_negative": int((x < 0).sum()),
        "n_unique": int(x.nunique()),
        "min": x.min(),
        "q01": x.quantile(0.01),
        "q25": x.quantile(0.25),
        "median": x.median(),
        "q75": x.quantile(0.75),
        "q99": x.quantile(0.99),
        "max": x.max(),
        "mean": x.mean(),
        "std": x.std()
    })

target_summary = pd.DataFrame(target_rows)

display(target_summary)

In [0]:
# ============================================================
# 8B. HYDROGEN TARGET SKEW DIAGNOSTIC
# ============================================================

if h2_col is not None:

    h2 = pd.to_numeric(
        db_ml[h2_col],
        errors="coerce"
    )

    valid_h2 = h2[h2 >= 0].dropna()

    print("Raw H2 skew:", valid_h2.skew())

    print(
        "log10(1 + H2) skew:",
        np.log10(1 + valid_h2).skew()
    )

## 8. Variable inventory and preliminary role classification

In [0]:
# ============================================================
# 9. PRELIMINARY VARIABLE ROLE CLASSIFICATION
# ============================================================

TARGETS = {
    "bandgap_energy",
    "hydrogen_production_rate"
}

IDENTIFIERS = {
    "compound",
    "doi_reference"
}

EXPERIMENTAL_KEYWORDS = [
    "light",
    "radiation",
    "particle",
    "surface",
    "catalyst",
    "co_catalyst",
    "reaction",
    "solution",
    "sacrificial",
    "carbon_source",
    "preparation",
    "calcination"
]


def classify_column(c):

    lc = str(c).strip().lower()

    if lc in TARGETS:
        return "target"

    if lc in IDENTIFIERS:
        return "identifier_metadata"

    if lc.endswith("_uom"):
        return "unit"

    if any(
        keyword in lc
        for keyword in EXPERIMENTAL_KEYWORDS
    ):
        return "experimental_or_reaction"

    return "raw_material_or_other"


role_inventory = pd.DataFrame({

    "column": db_ml.columns,

    "role": [
        classify_column(c)
        for c in db_ml.columns
    ],

    "dtype": [
        str(db_ml[c].dtype)
        for c in db_ml.columns
    ],

    "n_unique": [
        db_ml[c].nunique(dropna=True)
        for c in db_ml.columns
    ],

    "missing_pct": [
        100 * db_ml[c].isna().mean()
        for c in db_ml.columns
    ]
})

display(role_inventory)

## 9. Categorical variables

In [0]:
# ============================================================
# 10. CATEGORICAL VARIABLES
# ============================================================

categorical_cols = db_ml.select_dtypes(
    include=["object", "string", "category", "bool"]
).columns

categorical_summary_rows = []

for c in categorical_cols:

    categorical_summary_rows.append({
        "column": c,
        "dtype": str(db_ml[c].dtype),
        "n_unique": db_ml[c].nunique(dropna=True),
        "missing": db_ml[c].isna().sum(),
        "missing_pct": 100 * db_ml[c].isna().mean(),
        "top_values": str(
            db_ml[c]
            .value_counts(dropna=False)
            .head(10)
            .to_dict()
        )
    })

categorical_summary = pd.DataFrame(
    categorical_summary_rows
)

display(categorical_summary)

## 10. Numeric variables and suspicious values

In [0]:
# ============================================================
# 11. NUMERIC VARIABLE AUDIT
# ============================================================

numeric_cols = db_ml.select_dtypes(
    include=np.number
).columns

numeric_summary = (
    db_ml[numeric_cols]
    .describe()
    .T
    .reset_index()
    .rename(columns={"index": "column"})
)

display(numeric_summary)

In [0]:
# ============================================================
# 11B. NEGATIVE VALUES
# ============================================================

negative_rows = []

for c in numeric_cols:
    x = pd.to_numeric(db_ml[c], errors="coerce")

    n_negative = int((x < 0).sum())

    if n_negative > 0:
        negative_rows.append({
            "column": c,
            "negative_count": n_negative
        })

if len(negative_rows) > 0:

    negative_summary = pd.DataFrame(negative_rows)

    print("Columns containing negative values:")
    display(negative_summary)

else:

    print("No negative values were found in numeric columns.")

In [0]:
# ============================================================
# 11C. ZERO VALUES
# ============================================================

zero_rows = []

for c in numeric_cols:
    x = pd.to_numeric(db_ml[c], errors="coerce")

    n_zero = int((x == 0).sum())

    if n_zero > 0:
        zero_rows.append({
            "column": c,
            "zero_count": n_zero,
            "zero_pct": 100 * n_zero / len(x)
        })

if len(zero_rows) > 0:

    zero_summary = pd.DataFrame(zero_rows)

    print("Columns containing zero values:")
    display(zero_summary)

else:

    print("No zero values were found in numeric columns.")

## 12. Element-property table audit

In [0]:
# ============================================================
# 12A. ELEMENT COLUMN
# ============================================================

element_col = find_col(
    atom_props_spark,
    ["Element", "element"]
)

print("Element column:", element_col)

In [0]:
# ============================================================
# 12B. UNIQUE ELEMENTS
# ============================================================

if element_col is not None:

    unique_elements = (
        atom_props_spark
        .select(element_col)
        .where(
            F.col(element_col).isNotNull()
        )
        .distinct()
        .orderBy(element_col)
    )

    print(
        "Unique elements:",
        unique_elements.count()
    )

    display(unique_elements)

else:

    print("Element column not found.")

In [0]:
# ============================================================
# 12C. DUPLICATE ELEMENTS
# ============================================================

if element_col is not None:

    duplicate_element_rows = (
        atom_props_spark
        .groupBy(element_col)
        .count()
        .filter(
            F.col("count") > 1
        )
        .orderBy(
            F.desc("count")
        )
    )

    if duplicate_element_rows.limit(1).count() > 0:
        display(duplicate_element_rows)
    else:
        print("No duplicate element entries found.")

In [0]:
# ============================================================
# 12D. NUMERIC DESCRIPTORS
# ============================================================

atom_numeric_fields = [
    field
    for field in atom_props_spark.schema.fields
    if field.dataType.typeName() in [
        "double",
        "float",
        "integer",
        "long",
        "short",
        "decimal"
    ]
]

print(
    "Numeric descriptor columns:",
    len(atom_numeric_fields)
)

for field in atom_numeric_fields:
    print(
        f"{field.name}: {field.dataType}"
    )

## 13. Element coverage in db_ml

In [0]:
# ============================================================
# 13A. ELEMENT COVERAGE
# ============================================================

if (
    compound_col is not None
    and element_col is not None
):

    known_elements = {
        row[element_col]
        for row in (
            atom_props_spark
            .select(element_col)
            .where(
                F.col(element_col).isNotNull()
            )
            .distinct()
            .collect()
        )
    }

    observed_tokens = set()

    for formula in (
        db_ml[compound_col]
        .dropna()
        .astype(str)
    ):
        observed_tokens.update(
            re.findall(
                r"[A-Z][a-z]?",
                formula
            )
        )

    coverage = pd.DataFrame({
        "element": sorted(observed_tokens)
    })

    coverage["in_atom_props"] = (
        coverage["element"]
        .isin(known_elements)
    )

    print(
        "Observed element-like tokens:",
        len(coverage)
    )

    print(
        "Covered:",
        int(
            coverage["in_atom_props"].sum()
        )
    )

    print(
        "Missing:",
        int(
            (~coverage["in_atom_props"]).sum()
        )
    )

    if (~coverage["in_atom_props"]).any():

        print("Elements missing from atom_props:")

        display(
            coverage[
                ~coverage["in_atom_props"]
            ]
        )

    else:

        print(
            "All observed element-like tokens "
            "are present in atom_props."
        )

else:

    print(
        "Element coverage could not be calculated "
        "because Compound or Element was not found."
    )

## 14. Potential leakage / modeling-risk candidates

In [0]:
# ============================================================
# 14A. LEAKAGE / MODELING-RISK AUDIT
# ============================================================

risk_rows = []

for c in db_ml.columns:

    lc = str(c).lower()

    flags = []

    if lc in {
        "hydrogen_production_rate",
        "bandgap_energy"
    }:
        flags.append("target")

    if "doi" in lc:
        flags.append("metadata")

    if lc.endswith("_uom"):
        flags.append("unit_column")

    if any(
        keyword in lc
        for keyword in [
            "radiation",
            "catalyst",
            "co_catalyst",
            "reaction",
            "solution",
            "sacrificial",
            "preparation",
            "calcination"
        ]
    ):
        flags.append("experimental_condition")

    if "incident_light_cut_off" in lc:
        flags.append(
            "potential_target_proxy"
        )

    if flags:
        risk_rows.append({
            "column": c,
            "risk_flags": "; ".join(flags)
        })

if len(risk_rows) > 0:

    potential_risk = pd.DataFrame(
        risk_rows
    )

    display(potential_risk)

else:

    print(
        "No predefined leakage-risk candidates found."
    )

## 15. Constant and near-constant variables

In [0]:
# ============================================================
# 15A. CONSTANT / NEAR-CONSTANT VARIABLES
# ============================================================

constant_rows = []

for c in db_ml.columns:

    n_unique = db_ml[c].nunique(
        dropna=True
    )

    non_null = db_ml[c].notna().sum()

    if non_null == 0:

        dominant_pct = 100.0

    else:

        counts = (
            db_ml[c]
            .value_counts(dropna=True)
        )

        dominant_pct = (
            100 * counts.iloc[0] / non_null
        )

    constant_rows.append({
        "column": c,
        "n_unique": n_unique,
        "dominant_value_pct": dominant_pct,
        "constant": n_unique <= 1,
        "near_constant_gt_95pct":
            dominant_pct >= 95
    })

constant_audit = pd.DataFrame(
    constant_rows
)

display(
    constant_audit[
        constant_audit["constant"]
        |
        constant_audit[
            "near_constant_gt_95pct"
        ]
    ]
)

## 16. Final audit checklist

In [0]:
# ============================================================
# 16A. FINAL AUDIT CHECKLIST
# ============================================================

print("=" * 60)
print("DATA AUDIT COMPLETE")
print("=" * 60)

print(
    f"db_ml:       "
    f"{db_ml_rows:,} rows × {db_ml_cols} columns"
)

print(
    f"atom_props:  "
    f"{atom_props_rows:,} rows × {atom_props_cols} columns"
)

print()

print(
    f"Bandgap target present: "
    f"{bandgap_col is not None}"
)

print(
    f"Hydrogen target present: "
    f"{h2_col is not None}"
)

print(
    f"Compound identifier present: "
    f"{compound_col is not None}"
)

print(
    f"DOI metadata present: "
    f"{doi_col is not None}"
)

print()

print("Missingness audited: True")
print("Repeated experiments audited: True")
print("Negative values audited: True")
print("Zero values audited: True")
print("Element coverage audited: True")
print("Leakage candidates flagged: True")
print("Constant variables audited: True")

print()

print("Feature engineering performed: False")
print("Imputation performed: False")
print("Correlation filtering performed: False")
print("Machine learning performed: False")
print("Train/test split performed: False")

print()
print("Audit finished successfully.")

# Audit conclusion

This notebook is restricted to raw-data auditing.

The outputs from this notebook will be used to define the next-stage data preparation and feature-engineering strategy without performing any target-dependent preprocessing or machine-learning operations here.